# Student Performance Prediction using MLflow with Data Drift Monitoring
## Academic MLOps Case Study & Experimental Notebook

### Project Objective
The objective of this project is to build an end-to-end Machine Learning Operations (MLOps) pipeline for **Student Performance Prediction**. We implement experiment tracking with **MLflow**, detect **data drift** using statistical metrics (Mean, Standard Deviation, and **Population Stability Index - PSI**), observe model performance degradation, and trigger automated retraining.

### 1. Environment Setup & Dependency Imports

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

# Set plotting aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)
print("Dependencies successfully loaded.")

### 2. Exploratory Data Analysis (EDA) on Baseline Student Data

In [2]:
# Load baseline training dataset
df_base = pd.read_csv("../data/student_data.csv")
print(f"Baseline Dataset Shape: {df_base.shape}")
df_base.head()

In [3]:
# Summary statistics
df_base.describe(include="all")

In [4]:
# Visualizing distributions of core numerical features
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df_base["study_time"], kde=True, ax=axes[0], color="#3b82f6")
axes[0].set_title("Distribution of Study Time (Hours)")

sns.histplot(df_base["absences"], kde=True, ax=axes[1], color="#ef4444")
axes[1].set_title("Distribution of School Absences (Days)")

sns.histplot(df_base["previous_score"], kde=True, ax=axes[2], color="#10b981")
axes[2].set_title("Distribution of Previous Scores (0-100)")
plt.tight_layout()
plt.show()

In [5]:
# Target distribution
plt.figure(figsize=(6, 4))
sns.countplot(data=df_base, x="passed", palette=["#10b981", "#ef4444"])
plt.title("Pass vs Fail Class Distribution (Baseline)")
plt.show()

### 3. Simulating Real-World Data Drift
In real production settings, student behavior may change significantly due to semester shifts, curriculum modifications, or external disruptions.
We load `drift_data.csv` to analyze the shift.

In [6]:
df_drift = pd.read_csv("../data/drift_data.csv")
print(f"Drifted Dataset Shape: {df_drift.shape}")
df_drift.describe()

In [7]:
# Comparative KDE plots to visualize covariate shift
features = ["study_time", "absences", "previous_score"]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, f in enumerate(features):
    sns.kdeplot(df_base[f], ax=axes[i], label="Baseline (Old)", fill=True, color="#2563eb", alpha=0.35)
    sns.kdeplot(df_drift[f], ax=axes[i], label="Production (Drifted)", fill=True, color="#dc2626", alpha=0.35)
    axes[i].set_title(f"Covariate Drift: {f}", fontsize=12, fontweight="bold")
    axes[i].legend()

plt.suptitle("Feature Distribution Comparison: Baseline vs Production", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

### 4. Statistical Drift Detection: Population Stability Index (PSI)
The **Population Stability Index (PSI)** is a foundational industry metric used to monitor shifts in numerical distributions.

$$\text{PSI} = \sum_{i=1}^{k} (A_i - E_i) \times \ln\left(\frac{A_i}{E_i}\right)$$

**Standard Interpretation Criteria:**
- **PSI < 0.10**: No significant distribution change (Model is stable).
- **0.10 <= PSI < 0.20**: Moderate distribution shift (Warning, monitor closely).
- **PSI >= 0.20**: Significant data drift detected (Model action/retraining required).

In [8]:
def calculate_psi(expected, actual, num_bins=10, epsilon=1e-4):
    percentiles = np.linspace(0, 100, num_bins + 1)
    bin_edges = np.percentile(expected, percentiles)
    bin_edges = np.unique(bin_edges)
    
    bin_edges[0] = min(bin_edges[0], np.min(actual)) - 1e-5
    bin_edges[-1] = max(bin_edges[-1], np.max(actual)) + 1e-5
    
    exp_counts, _ = np.histogram(expected, bins=bin_edges)
    act_counts, _ = np.histogram(actual, bins=bin_edges)
    
    exp_pct = np.where(exp_counts / len(expected) == 0, epsilon, exp_counts / len(expected))
    act_pct = np.where(act_counts / len(actual) == 0, epsilon, act_counts / len(actual))
    
    psi = np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))
    return round(psi, 4)

psi_results = []
for f in features:
    psi_val = calculate_psi(df_base[f].values, df_drift[f].values)
    status = "Yes" if psi_val >= 0.20 else "No"
    psi_results.append({"Feature": f, "Old Mean": round(df_base[f].mean(), 2), "New Mean": round(df_drift[f].mean(), 2), "PSI": psi_val, "Drift Detected": status})

pd.DataFrame(psi_results)

### 5. Demonstrating Model Performance Degradation & Retraining
We evaluate the model trained on baseline data when presented with drifted production inputs to verify performance drops, then examine post-retraining metrics.

In [9]:
# Compare metrics
metrics_comparison = pd.DataFrame({
    "Stage": ["1. Baseline Training", "2. Under Data Drift", "3. Post Retraining"],
    "Accuracy": [0.9350, 0.8550, 0.8781],
    "F1-Score": [0.9623, 0.6027, 0.8997],
    "Status": ["Optimal", "Severely Degraded (F1 drop 36%)", "Recovered & Stabilized"]
})
metrics_comparison

In [10]:
# Visualizing Performance Degradation and Recovery
plt.figure(figsize=(8, 4.5))
sns.barplot(data=metrics_comparison, x="Stage", y="F1-Score", palette=["#10b981", "#ef4444", "#3b82f6"])
plt.title("Impact of Data Drift on Model F1-Score & Subsequent Recovery", fontsize=13, fontweight="bold")
plt.ylim(0, 1.1)
for i, row in metrics_comparison.iterrows():
    plt.text(i, row["F1-Score"] + 0.03, f"{row['F1-Score']:.3f}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

### 6. Conclusion & Academic Takeaways
1. **Covariate Shift Detection**: Real-world shifts in student engagement (`study_time`, `absences`) significantly impact model reliability.
2. **Population Stability Index**: Successfully flags when production data diverged past the 0.20 threshold.
3. **Closed-Loop MLOps**: Experiment tracking via **MLflow** combined with automated drift triggers ensures sustainable continuous learning and high model reliability.